# Pandas  pour un Data Scientist

Ce cours couvre les bases de la bibliothèque Pandas en Python, qui est un outil incontournable pour la manipulation et l'analyse de données en data science. Pandas fournit des structures comme les DataFrames et les Series pour travailler avec des données tabulaires, similaires à Excel ou SQL. Nous nous concentrerons sur les commandes clés qu'un data scientist doit connaître : création, lecture/écriture, sélection, manipulation, regroupement, fusion, et gestion des données manquantes. Nous verrons des exemples d'utilisation et quelques exercices pour pratiquer.

En Python, importez Pandas avec `import pandas as pd`. Les exemples supposent que vous avez des données de test (installez Pandas via `pip install pandas` si nécessaire). Pandas s'intègre bien avec NumPy pour les calculs numériques.



## 1. Création de DataFrames et Series
Un `DataFrame` est une table 2D avec des colonnes nommées ; une `Series` est une colonne 1D.

### Exemples
```python
import pandas as pd
import numpy as np  # Souvent utilisé avec Pandas

# Création d'une Series
serie = pd.Series([1, 2, 3, np.nan], index=['a', 'b', 'c', 'd'])
print(serie)  # Affiche la Series avec index

# Création d'un DataFrame
data = {'Nom': ['Alice', 'Bob', 'Charlie'],
        'Âge': [25, 30, 35],
        'Ville': ['Paris', 'Lyon', 'Marseille']}
df = pd.DataFrame(data)
print(df)  # Affiche le DataFrame

# À partir d'un dictionnaire ou d'une liste de listes
df2 = pd.DataFrame([[1, 2], [3, 4]], columns=['Col1', 'Col2'])
```

Note : Utilisez `index=` pour personnaliser les lignes. `pd.DataFrame.from_dict()` pour des options avancées.



## 2. Lecture et Écriture de Données
Pandas supporte CSV, Excel, SQL, JSON, etc.

### Exemples
```python
# Lecture d'un CSV
df = pd.read_csv('fichier.csv', sep=',', encoding='utf-8')  # Options : header=None, usecols=['col1', 'col2']

# Lecture Excel
df_excel = pd.read_excel('fichier.xlsx', sheet_name='Feuille1')

# Écriture
df.to_csv('sortie.csv', index=False)  # Sans index
df.to_excel('sortie.xlsx', index=False)

# Aperçu des données
print(df.head(5))  # Premières 5 lignes
print(df.tail(3))  # Dernières 3 lignes
print(df.info())  # Infos sur types et mémoire
print(df.describe())  # Statistiques descriptives
```

Note : Pour les gros fichiers, utilisez `chunksize` dans `read_csv` pour charger par lots. Gérez les dates avec `parse_dates=['col_date']`.



## 3. Sélection et Filtrage de Données
Accédez aux données avec `loc` (labels), `iloc` (indices), ou directement.

### Exemples
```python
# Sélection par colonne
noms = df['Nom']  # Series
subset = df[['Nom', 'Âge']]  # DataFrame

# Par ligne (iloc pour indices)
premiere_ligne = df.iloc[0]
lignes_1_a_2 = df.iloc[1:3]

# Par label (loc)
alice = df.loc[0]  # Première ligne
filtre = df.loc[df['Âge'] > 28]  # Filtrage booléen

# Conditions multiples
filtre_complexe = df[(df['Âge'] > 25) & (df['Ville'] == 'Paris')]

# At pour valeur unique
print(df.at[0, 'Nom'])  # "Alice"
```

Note : Évitez les chaînages comme `df['col'][0]` pour des performances ; préférez `loc` ou `iloc`. Utilisez `query` pour des filtres SQL-like : `df.query("Âge > 25")`.



## 4. Manipulation de Données
Modifiez, appliquez des fonctions, triez, etc.

### Exemples
```python
# Ajout de colonne
df['Senior'] = df['Âge'] > 30  # Booléen

# Apply sur colonne
df['Nom_Maj'] = df['Nom'].apply(lambda x: x.upper())

# Map sur Series
df['Ville_Code'] = df['Ville'].map({'Paris': 75, 'Lyon': 69})

# Tri
df_trie = df.sort_values(by='Âge', ascending=False)

# Suppression
df_sans_col = df.drop(columns=['Ville'])
df_sans_ligne = df.drop(index=0)

# Renommage
df_renomme = df.rename(columns={'Nom': 'Prénom'})
```

Note : Pour des opérations vectorisées, utilisez NumPy-like : `df['Âge'] += 1`. `replace` pour substitutions : `df.replace(np.nan, 0)`.



## 5. Regroupement et Agrégation (GroupBy)
Groupez pour des stats par catégorie.

### Exemples
```python
# GroupBy simple
groupe = df.groupby('Ville')
print(groupe['Âge'].mean())  # Moyenne d'âge par ville

# Agrégations multiples
agg = groupe.agg({'Âge': ['mean', 'min'], 'Nom': 'count'})
print(agg)

# Transform (applique à chaque groupe)
df['Âge_Moyen_Ville'] = groupe['Âge'].transform('mean')
```

Note : `groupby` retourne un GroupBy object ; appliquez `mean()`, `sum()`, `count()`, etc. Utilisez `as_index=False` pour ne pas indexer par groupe.



## 6. Fusion et Jointure (Merge, Join, Concat)
Combinez des DataFrames.

### Exemples
```python
# Merge (comme SQL JOIN)
df1 = pd.DataFrame({'ID': [1, 2], 'Nom': ['Alice', 'Bob']})
df2 = pd.DataFrame({'ID': [1, 2], 'Score': [90, 85]})
merge = pd.merge(df1, df2, on='ID', how='inner')  # inner, left, right, outer

# Concat (empiler)
concat = pd.concat([df1, df2], axis=0)  # Lignes ; axis=1 pour colonnes

# Join (sur index)
df1.set_index('ID', inplace=True)
df2.set_index('ID', inplace=True)
join = df1.join(df2)
```

Note : `how='left'` conserve toutes les lignes de gauche. Utilisez `suffixes` pour conflits de colonnes.



## 7. Gestion des Données Manquantes
Détectez et traitez les NaN.

### Exemples
```python
# Détection
print(df.isnull().sum())  # Nombre de NaN par colonne
print(df.notnull())  # Booléen

# Suppression
df_clean = df.dropna()  # Supprime lignes avec NaN
df_clean_col = df.dropna(axis=1)  # Colonnes

# Remplissage
df_rempli = df.fillna(0)  # Avec 0
df_moyenne = df['Âge'].fillna(df['Âge'].mean())  # Avec moyenne
df_interpole = df.interpolate()  # Interpolation linéaire
```

Note : Utilisez `dropna(thresh=2)` pour garder les lignes avec au moins 2 non-NaN. `isna()` est alias de `isnull()`.



## 8. Visualisation Basique et Autres Commandes Utiles
Pandas intègre Matplotlib pour des plots simples.

### Exemples
```python
# Plot
df.plot(kind='bar', x='Nom', y='Âge')  # Barre

# Pivot table
pivot = df.pivot_table(values='Âge', index='Ville', aggfunc='mean')

# Échantillonnage
echantillon = df.sample(n=2)  # 2 lignes aléatoires

# Types de données
df['Âge'] = df['Âge'].astype(float)  # Conversion
```

Note : Importez `matplotlib.pyplot as plt` pour plus de personnalisation. Utilisez `value_counts()` pour comptages : `df['Ville'].value_counts()`.



## Exercices
Voici quelques exercices simples pour pratiquer. Essayez de les résoudre dans un interpréteur Python (avec des données fictives), puis vérifiez les résultats.

1. **Exercice 1 : Création et Aperçu**
   Créez un DataFrame avec des colonnes 'Produit', 'Prix', 'Quantité'. Lisez-le comme si c'était un CSV, affichez `head()` et `describe()`.




2. **Exercice 2 : Sélection et Filtrage**
   Dans un DataFrame de ventes, sélectionnez les lignes où 'Prix' > 50 avec `loc`. Ajoutez une colonne 'Total' = Prix * Quantité.



3. **Exercice 3 : GroupBy**
   Groupez par 'Produit' et calculez la somme des 'Quantité' et la moyenne des 'Prix'. Utilisez `agg` pour plusieurs fonctions.



4. **Exercice 4 : Fusion**
   Créez deux DataFrames (clients et commandes) et mergez-les sur 'ID_Client' avec how='left'. Gérez les NaN en remplissant avec 0.



5. **Exercice 5 : Manipulation Avancée**
   Dans un DataFrame avec des NaN, supprimez les lignes avec plus de 1 NaN. Triez par 'Prix' descendant, et plottez un histogramme des 'Quantité'.